In [ ]:
import sqlite3
import random
from datetime import datetime, timedelta

DB_NAME = "immigrant_support.db"
NUM_CLIENTS = 250
NUM_REFERRALS = 1200
random.seed(42)

def setup_database():
    conn = sqlite3.connect(DB_NAME)
    try:
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")

        cursor.executescript("""
        DROP TABLE IF EXISTS raw_referral_import;
        DROP TABLE IF EXISTS referral_tags;
        DROP TABLE IF EXISTS referrals;
        DROP TABLE IF EXISTS service_providers;
        DROP TABLE IF EXISTS clients;

        CREATE TABLE clients (
            client_id INTEGER PRIMARY KEY AUTOINCREMENT,
            region TEXT NOT NULL,
            age_years REAL CHECK(age_years IS NULL OR age_years >= 0),
            primary_language TEXT NOT NULL,
            entry_year INTEGER CHECK(entry_year BETWEEN 1900 AND 2100)
        );

        CREATE TABLE service_providers (
            provider_id INTEGER PRIMARY KEY AUTOINCREMENT,
            provider_name TEXT NOT NULL,
            provider_type TEXT NOT NULL CHECK(
                provider_type IN ('Healthcare', 'Legal', 'Education', 'Housing', 'Employment')
            )
        );

        CREATE TABLE referrals (
            referral_id INTEGER PRIMARY KEY AUTOINCREMENT,
            client_id INTEGER NOT NULL,
            provider_id INTEGER NOT NULL,
            referral_date TEXT NOT NULL,
            priority_level INTEGER NOT NULL CHECK(priority_level BETWEEN 1 AND 5),
            status TEXT NOT NULL DEFAULT 'Pending'
                CHECK(status IN ('Pending', 'In progress', 'Completed', 'Cancelled')),
            support_hours REAL CHECK(support_hours IS NULL OR support_hours >= 0),
            estimated_cost REAL CHECK(estimated_cost IS NULL OR estimated_cost >= 0),
            FOREIGN KEY (client_id) REFERENCES clients(client_id),
            FOREIGN KEY (provider_id) REFERENCES service_providers(provider_id)
        );

        CREATE TABLE referral_tags (
            referral_id INTEGER NOT NULL,
            tag TEXT NOT NULL,
            PRIMARY KEY (referral_id, tag),
            FOREIGN KEY (referral_id) REFERENCES referrals(referral_id)
        );

        CREATE TABLE raw_referral_import (
            import_id INTEGER PRIMARY KEY AUTOINCREMENT,
            client_id INTEGER,
            referral_date TEXT,
            notes TEXT
        );
        """)

        providers = [
            ("NHS Trust North", "Healthcare"),
            ("Legal Aid Collective", "Legal"),
            ("City Language School", "Education"),
            ("Council Housing Advice", "Housing"),
            ("Skills Match UK", "Employment"),
        ]
        cursor.executemany(
            "INSERT INTO service_providers (provider_name, provider_type) VALUES (?, ?)",
            providers,
        )

        regions = ["London", "West Midlands", "North West", "South East", "Scotland"]
        languages = ["Arabic", "Spanish", "Pashto", "Ukrainian", "French", "English"]
        client_ids = []

        for _ in range(NUM_CLIENTS):
            age = round(random.uniform(18, 75), 1) if random.random() > 0.05 else None
            cursor.execute(
                """INSERT INTO clients (region, age_years, primary_language, entry_year)
                   VALUES (?, ?, ?, ?)""",
                (
                    random.choice(regions),
                    age,
                    random.choice(languages),
                    random.randint(2015, 2024),
                ),
            )
            client_ids.append(cursor.lastrowid)

        start_date = datetime(2023, 1, 1)
        for i in range(NUM_REFERRALS):
            client_id = random.choice(client_ids)
            provider_id = random.randint(1, len(providers))
            referral_date = (
                start_date + timedelta(days=random.randint(0, 600))
            ).strftime("%Y-%m-%d")
            priority = random.randint(1, 5)
            status = random.choice(["Pending", "In progress", "Completed", "Cancelled"])
            hours = round(random.uniform(1, 20), 1) if random.random() > 0.10 else None
            cost = round(random.uniform(50, 500), 2) if random.random() > 0.05 else None

            cursor.execute(
                """INSERT INTO referrals
                   (client_id, provider_id, referral_date, priority_level, status,
                    support_hours, estimated_cost)
                   VALUES (?, ?, ?, ?, ?, ?, ?)""",
                (client_id, provider_id, referral_date, priority, status, hours, cost),
            )
            referral_id = cursor.lastrowid

            cursor.execute(
                "INSERT INTO referral_tags (referral_id, tag) VALUES (?, ?)",
                (referral_id, "Support"),
            )
            if random.random() > 0.5:
                cursor.execute(
                    "INSERT INTO referral_tags (referral_id, tag) VALUES (?, ?)",
                    (referral_id, "Follow-up"),
                )

            # Deliberately repeated staging rows for data-cleaning practice.
            if i < 20:
                for _ in range(2):
                    cursor.execute(
                        """INSERT INTO raw_referral_import
                           (client_id, referral_date, notes)
                           VALUES (?, ?, ?)""",
                        (client_id, referral_date, "Standard referral"),
                    )

        conn.commit()
        print(f"Created {DB_NAME} with {NUM_REFERRALS} referrals.")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

setup_database()


Created immigrant_support.db with 1200 referrals.


In [ ]:
from google.colab import files
files.download("immigrant_support.db")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>